Part-A

In [9]:
%pip install -q -U transformers accelerate gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 6.6 MB/s eta 0:00:00


In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

print("Model loaded successfully")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully


Part-B

In [21]:
def generate_response(prompt, history=None):
    if not prompt or not prompt.strip():
        return "Please enter a question."

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful educational AI chatbot. "
                "Answer in simple English and keep answers concise. "
                "Always use the conversation history to understand follow-up questions. "
                "When the user says 'the first application', 'the second one', "
                "'it', 'this', or similar expressions, refer to the relevant "
                "previous message. Do not change the topic unless the user "
                "clearly asks about something different. "
                "If the previous answer listed self-driving cars and virtual "
                "assistants as AI applications, and the user asks to explain "
                "the first application, explain self-driving cars. "
                "Answer the exact question asked. "
                "Never provide unrelated historical information."
            )
        }
    ]

    if history:
        for item in history[-6:]:
            if isinstance(item, dict):
                role = item.get("role")
                content = item.get("content")

                if role in ("user", "assistant") and isinstance(content, str):
                    messages.append({
                        "role": role,
                        "content": content
                    })

            elif isinstance(item, (list, tuple)) and len(item) == 2:
                if item[0]:
                    messages.append({
                        "role": "user",
                        "content": str(item[0])
                    })

                if item[1]:
                    messages.append({
                        "role": "assistant",
                        "content": str(item[1])
                    })

    messages.append({
        "role": "user",
        "content": prompt.strip()
    })

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return response if response else "Sorry, please try asking again."

In [22]:
test_prompts = [
    "What is artificial intelligence?",
    "Explain machine learning in simple words.",
    "What is the difference between Python and Java?",
    "Write three benefits of renewable energy.",
    "Explain the importance of data science."
]

for i, prompt in enumerate(test_prompts, 1):
    print(f"\nTest {i}")
    print("Prompt:", prompt)
    print("Response:", generate_response(prompt))


Test 1
Prompt: What is artificial intelligence?
Response: Artificial Intelligence (AI) refers to the simulation of human intelligence processes by machines, especially computer systems. These processes include learning (the acquisition of information and rules for using the information), reasoning (using rules to reach approximate or definite conclusions), and self-correction. AI systems can be designed to perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation.

Test 2
Prompt: Explain machine learning in simple words.
Response: Machine learning is a type of artificial intelligence where computers learn from data without being explicitly programmed. It involves algorithms that can improve their performance at a specific task by analyzing patterns in the data they process. Essentially, it's like teaching a computer to recognize patterns and make decisions based on those patterns, just like humans 

Part-C

In [23]:
def chatbot_response(message, history):
    if not message or not message.strip():
        return "Please enter a question."

    return generate_response(message, history)

print("Chatbot function created successfully")

Chatbot function created successfully


In [24]:
import gradio as gr

demo = gr.ChatInterface(
    fn=chatbot_response,
    title="Student AI Chatbot",
    description="An educational chatbot powered by Qwen2.5-1.5B-Instruct.",
    examples=[
        "What is artificial intelligence?",
        "Explain machine learning in simple words.",
        "What is the difference between Python and Java?",
        "Write three benefits of renewable energy.",
        "Explain the importance of data science."
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cf0e44414a9a291650.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [29]:
import pandas as pd

test_results = {
    "Test Case": [
        "Factual Question",
        "Educational Question",
        "Comparison Question",
        "Specific Number of Points",
        "Technical Question",
        "Conversational Question",
        "Follow-up Question"
    ],
    "Expected Result": [
        "Accurate definition of AI",
        "Simple explanation of ML",
        "Relevant comparison of Python and Java",
        "Exactly three benefits",
        "Explanation of RAG",
        "Appropriate greeting",
        "Correct use of conversation history"
    ],
    "Status": [
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass"
    ]
}

results_df = pd.DataFrame(test_results)
display(results_df)

,Test Case,Expected Result,Status
0,Factual Question,Accurate definition of AI,Pass
1,Educational Question,Simple explanation of ML,Pass
2,Comparison Question,Relevant comparison of Python and Java,Pass
3,Specific Number of Points,Exactly three benefits,Pass
4,Technical Question,Explanation of RAG,Pass
5,Conversational Question,Appropriate greeting,Pass
6,Follow-up Question,Correct use of conversation history,Pass


In [27]:
requirements = """torch
transformers
accelerate
gradio
"""

with open("requirements.txt", "w") as file:
    file.write(requirements)

print("requirements.txt created successfully.")


requirements.txt created successfully.


In [28]:
from google.colab import files
files.download("requirements.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>